# ADVANCED EDA & PRICING ANALYSIS

In [2]:
import pandas as pd
import numpy as np


In [3]:
sales_df = pd.read_csv("pricing_sales_clean.csv")

C:\Users\Dell\AppData\Local\Temp\ipykernel_988\2627056161.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv("pricing_sales_clean.csv")


In [4]:
# ============================================================
# 1. BASIC BUSINESS KPIs
# ============================================================

total_revenue = sales_df["Revenue"].sum()
total_units = sales_df["Quantity"].sum()
total_orders = sales_df["Invoice_ID"].nunique()
total_products = sales_df["Stock_Code"].nunique()
total_customers = sales_df["Customer_ID"].nunique()

average_order_value = total_revenue / total_orders
average_units_per_order = total_units / total_orders
average_price = sales_df["Unit_Price"].mean()

print("========== BUSINESS KPIs ==========")

print(f"Total Revenue          : £{total_revenue:,.2f}")
print(f"Total Units Sold       : {total_units:,.0f}")
print(f"Total Orders           : {total_orders:,}")
print(f"Unique Products        : {total_products:,}")
print(f"Unique Customers       : {total_customers:,}")
print(f"Average Order Value    : £{average_order_value:,.2f}")
print(f"Avg Units per Order    : {average_units_per_order:,.2f}")
print(f"Average Unit Price     : £{average_price:,.2f}")

========== BUSINESS KPIs ==========
Total Revenue          : £10,272,136.23
Total Units Sold       : 5,812,948
Total Orders           : 20,953
Unique Products        : 4,250
Unique Customers       : 4,312
Average Order Value    : £490.25
Avg Units per Order    : 277.43
Average Unit Price     : £4.27


In [5]:
# ============================================================
# 2. PRICE BAND PERFORMANCE
# ============================================================

price_band_analysis = (
    sales_df
    .groupby("Price_Band", observed=True)
    .agg(
        Units_Sold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Orders=("Invoice_ID", "nunique"),
        Products=("Stock_Code", "nunique")
    )
    .reset_index()
)

price_band_analysis["Revenue_Per_Unit"] = (
    price_band_analysis["Revenue"] /
    price_band_analysis["Units_Sold"]
)

price_band_analysis["Revenue_Share_%"] = (
    price_band_analysis["Revenue"] /
    total_revenue * 100
)

display(price_band_analysis)

,Price_Band,Units_Sold,Revenue,Orders,Products,Revenue_Per_Unit,Revenue_Share_%
0,1-2,1814212,2675816.070,17023,1855,1.474919,26.049266
1,10-25,53860,706311.940,7557,698,13.113850,6.875999
2,2-5,1098400,3397038.670,17609,2522,3.092715,33.070421
3,25-50,5166,158900.040,1312,161,30.758815,1.546904
4,5-10,239625,1684403.740,13774,1521,7.029332,16.397794
5,50+,1133,444942.830,894,48,392.712118,4.331551
6,<=1,2600552,1204722.944,14465,1258,0.463257,11.728066


In [6]:
# ============================================================
# 3. PRICE VS DEMAND
# ============================================================

price_demand = (
    sales_df
    .groupby("Price_Band", observed=True)
    .agg(
        Average_Price=("Unit_Price", "mean"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Orders=("Invoice_ID", "nunique"),
        Products=("Stock_Code", "nunique")
    )
    .reset_index()
)

price_demand["Units_Per_Product"] = (
    price_demand["Units_Sold"] /
    price_demand["Products"]
)

price_demand["Revenue_Per_Product"] = (
    price_demand["Revenue"] /
    price_demand["Products"]
)

display(price_demand)

,Price_Band,Average_Price,Units_Sold,Revenue,Orders,Products,Units_Per_Product,Revenue_Per_Product
0,1-2,1.517655,1814212,2675816.070,17023,1855,978.011860,1442.488447
1,10-25,13.849375,53860,706311.940,7557,698,77.163324,1011.908223
2,2-5,3.252060,1098400,3397038.670,17609,2522,435.527359,1346.962201
3,25-50,31.433247,5166,158900.040,1312,161,32.086957,986.956770
4,5-10,7.183379,239625,1684403.740,13774,1521,157.544379,1107.431782
5,50+,447.123680,1133,444942.830,894,48,23.604167,9269.642292
6,<=1,0.649024,2600552,1204722.944,14465,1258,2067.211447,957.649399


In [7]:
# ============================================================
# 4. PRICE-DEMAND CORRELATION
# ============================================================

price_demand_corr = sales_df[
    ["Unit_Price", "Quantity"]
].corr()

display(price_demand_corr)

,Unit_Price,Quantity
Unit_Price,1.000000,-0.005159
Quantity,-0.005159,1.000000


In [8]:
# Convert Invoice_Date to datetime
sales_df["Invoice_Date"] = pd.to_datetime(
    sales_df["Invoice_Date"],
    errors="coerce"
)

# Check datatype
print(sales_df["Invoice_Date"].dtype)

# Check if any dates became missing
print("Missing dates:", sales_df["Invoice_Date"].isna().sum())

datetime64[ns]
Missing dates: 0


In [9]:
# ============================================================
# 5. PRODUCT-LEVEL PRICE ELASTICITY
# STEP 5.1 — MONTHLY PRODUCT DATA
# ============================================================

monthly_product = (
    sales_df
    .assign(
        Month=sales_df["Invoice_Date"].dt.to_period("M")
    )
    .groupby(
        ["Stock_Code", "Month"],
        observed=True
    )
    .agg(
        Average_Price=("Unit_Price", "mean"),
        Quantity_Sold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Orders=("Invoice_ID", "nunique")
    )
    .reset_index()
)

display(monthly_product.head(10))

,Stock_Code,Month,Average_Price,Quantity_Sold,Revenue,Orders
0,10002,2009-12,0.977500,215,185.30,20
1,10002,2010-01,0.945294,291,248.97,17
2,10002,2010-02,0.958000,257,220.07,15
3,10002,2010-03,0.911111,641,493.09,17
4,10002,2010-04,0.924167,1132,843.73,24
5,10002,2010-05,0.953824,1465,1157.49,33
6,10002,2010-06,0.999730,463,404.20,37
7,10002,2010-07,0.969615,512,416.49,25
8,10002,2010-08,1.014783,586,450.15,23
9,10002,2010-09,0.971500,240,213.72,20


In [10]:
# ============================================================
# STEP 5.2 — COMPARE WITH PREVIOUS MONTH
# ============================================================

# First sort the data
monthly_product = monthly_product.sort_values(
    ["Stock_Code", "Month"]
)

# Get previous month's price
monthly_product["Previous_Price"] = (
    monthly_product.groupby("Stock_Code")["Average_Price"].shift(1)
)

# Get previous month's quantity
monthly_product["Previous_Quantity"] = (
    monthly_product.groupby("Stock_Code")["Quantity_Sold"].shift(1)
)

# Calculate price percentage change
monthly_product["Price_Change_%"] = (
    (monthly_product["Average_Price"] -
     monthly_product["Previous_Price"])
    / monthly_product["Previous_Price"]
) * 100

# Calculate quantity percentage change
monthly_product["Quantity_Change_%"] = (
    (monthly_product["Quantity_Sold"] -
     monthly_product["Previous_Quantity"])
    / monthly_product["Previous_Quantity"]
) * 100

# Show the result
display(
    monthly_product[
        [
            "Stock_Code",
            "Month",
            "Average_Price",
            "Previous_Price",
            "Price_Change_%",
            "Quantity_Sold",
            "Previous_Quantity",
            "Quantity_Change_%"
        ]
    ].head(15)
)

,Stock_Code,Month,Average_Price,Previous_Price,Price_Change_%,Quantity_Sold,Previous_Quantity,Quantity_Change_%
0,10002,2009-12,0.977500,NaN,NaN,215,NaN,NaN
1,10002,2010-01,0.945294,0.977500,-3.294719,291,215.0,35.348837
2,10002,2010-02,0.958000,0.945294,1.344119,257,291.0,-11.683849
3,10002,2010-03,0.911111,0.958000,-4.894456,641,257.0,149.416342
4,10002,2010-04,0.924167,0.911111,1.432927,1132,641.0,76.599064
5,10002,2010-05,0.953824,0.924167,3.209038,1465,1132.0,29.416961
6,10002,2010-06,0.999730,0.953824,4.812861,463,1465.0,-68.395904
7,10002,2010-07,0.969615,0.999730,-3.012249,512,463.0,10.583153
8,10002,2010-08,1.014783,0.969615,4.658262,586,512.0,14.453125
9,10002,2010-09,0.971500,1.014783,-4.265210,240,586.0,-59.044369


In [11]:
# ============================================================
# STEP 5.3 — CALCULATE PRICE ELASTICITY
# ============================================================

monthly_product["Elasticity"] = (
    monthly_product["Quantity_Change_%"]
    / monthly_product["Price_Change_%"]
)

display(
    monthly_product[
        [
            "Stock_Code",
            "Month",
            "Price_Change_%",
            "Quantity_Change_%",
            "Elasticity"
        ]
    ].head(15)
)

,Stock_Code,Month,Price_Change_%,Quantity_Change_%,Elasticity
0,10002,2009-12,NaN,NaN,NaN
1,10002,2010-01,-3.294719,35.348837,-10.728937
2,10002,2010-02,1.344119,-11.683849,-8.692567
3,10002,2010-03,-4.894456,149.416342,-30.527671
4,10002,2010-04,1.432927,76.599064,53.456368
5,10002,2010-05,3.209038,29.416961,9.166909
6,10002,2010-06,4.812861,-68.395904,-14.211070
7,10002,2010-07,-3.012249,10.583153,-3.513373
8,10002,2010-08,4.658262,14.453125,3.102686
9,10002,2010-09,-4.265210,-59.044369,13.843250


In [12]:
# ============================================================
# STEP 5.4 — CLEAN ELASTICITY VALUES
# ============================================================

# Remove rows where elasticity cannot be calculated
elasticity_df = monthly_product.dropna(
    subset=["Elasticity"]
).copy()

# Remove infinite values
elasticity_df = elasticity_df[
    np.isfinite(elasticity_df["Elasticity"])
].copy()

# Keep only reasonable elasticity values
elasticity_df = elasticity_df[
    elasticity_df["Elasticity"].between(-10, 10)
].copy()

print("========== ELASTICITY DATA ==========")

print(f"Original monthly rows : {len(monthly_product):,}")
print(f"Clean elasticity rows : {len(elasticity_df):,}")

display(
    elasticity_df[
        [
            "Stock_Code",
            "Month",
            "Price_Change_%",
            "Quantity_Change_%",
            "Elasticity"
        ]
    ].head(15)
)

========== ELASTICITY DATA ==========
Original monthly rows : 35,425
Clean elasticity rows : 15,190


,Stock_Code,Month,Price_Change_%,Quantity_Change_%,Elasticity
2,10002,2010-02,1.344119,-11.683849,-8.692567
5,10002,2010-05,3.209038,29.416961,9.166909
7,10002,2010-07,-3.012249,10.583153,-3.513373
8,10002,2010-08,4.658262,14.453125,3.102686
11,10002,2010-11,6.007144,-7.751196,-1.290330
12,10002,2010-12,28.566434,-83.402490,-2.919598
21,10120,2010-01,-16.256158,-72.173913,4.439789
22,10120,2010-02,51.764706,-84.375000,-1.629972
33,10123C,2010-01,14.775846,-78.873239,-5.337985
34,10123C,2010-02,-32.642487,-56.666667,1.735979


In [13]:
# ============================================================
# STEP 5.5 — AVERAGE ELASTICITY BY PRODUCT
# ============================================================

product_elasticity = (
    elasticity_df
    .groupby("Stock_Code")
    .agg(
        Average_Elasticity=("Elasticity", "mean"),
        Average_Price=("Average_Price", "mean"),
        Total_Quantity=("Quantity_Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Number_of_Months=("Month", "nunique")
    )
    .reset_index()
)

# Show products with the most reliable observations first
product_elasticity = product_elasticity.sort_values(
    "Number_of_Months",
    ascending=False
)

display(product_elasticity.head(20))

,Stock_Code,Average_Elasticity,Average_Price,Total_Quantity,Total_Revenue,Number_of_Months
3276,DOT,0.216521,172.396849,681,97834.13,12
3278,POST,-0.317668,26.362858,2185,46552.59,12
3006,85216,-0.187747,1.961057,2559,1873.03,12
731,21533,-0.296496,5.445326,1870,9296.93,12
816,21657,-0.743393,8.668368,223,1715.81,12
2060,40016,-0.482552,1.728526,9866,3337.10,12
857,21700,-1.625100,2.777797,4020,4113.12,12
2015,37370,-0.576954,5.535967,4551,7057.41,12
177,20724,0.226913,1.072685,13893,13051.79,12
1068,21983,-0.643135,0.380495,3778,1087.84,12


In [14]:
# Step 5.6 - Classify Price Elasticity

product_elasticity["Elasticity_Type"] = np.where(
    product_elasticity["Average_Elasticity"] < -1,
    "Elastic",
    np.where(
        product_elasticity["Average_Elasticity"] < 0,
        "Inelastic",
        np.where(
            product_elasticity["Average_Elasticity"] == -1,
            "Unit Elastic",
            "Positive / Unusual"
        )
    )
)

display(
    product_elasticity[
        [
            "Stock_Code",
            "Average_Elasticity",
            "Average_Price",
            "Total_Quantity",
            "Total_Revenue",
            "Number_of_Months",
            "Elasticity_Type"
        ]
    ].head(20)
)

,Stock_Code,Average_Elasticity,Average_Price,Total_Quantity,Total_Revenue,Number_of_Months,Elasticity_Type
3276,DOT,0.216521,172.396849,681,97834.13,12,Positive / Unusual
3278,POST,-0.317668,26.362858,2185,46552.59,12,Inelastic
3006,85216,-0.187747,1.961057,2559,1873.03,12,Inelastic
731,21533,-0.296496,5.445326,1870,9296.93,12,Inelastic
816,21657,-0.743393,8.668368,223,1715.81,12,Inelastic
2060,40016,-0.482552,1.728526,9866,3337.10,12,Inelastic
857,21700,-1.625100,2.777797,4020,4113.12,12,Elastic
2015,37370,-0.576954,5.535967,4551,7057.41,12,Inelastic
177,20724,0.226913,1.072685,13893,13051.79,12,Positive / Unusual
1068,21983,-0.643135,0.380495,3778,1087.84,12,Inelastic


In [15]:
product_elasticity.to_csv(
    "product_elasticity.csv",
    index=False
)